## ArcHotel AI Agents

ArcHotel AI Agents is an interactive AI-powered hotel booking assistant that helps users search, explore, and reserve hotels across multiple cities such as Paris, New York, and Dubai. The agent is powered by OpenAI’s GPT-4.1-mini model and integrates with a SQLite database to provide realistic hotel and room availability, pricing, and amenities.

## Features

- Search hotels by city, stay dates, budget, star rating, and amenities

- View hotel details including available room types and pricing

- Make room reservations with a generated confirmation code

- Conversational interface that guides users on adjusting budget, dates, or preferences

- Gradio-based web interface with authentication

In [35]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
from datetime import datetime
import random
import string

In [36]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [37]:
DB = "arc_hotels.db"


In [ ]:

conn = sqlite3.connect(DB)
cursor = conn.cursor()


# Hotels table
cursor.execute("""
CREATE TABLE IF NOT EXISTS hotels (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    city TEXT,
    price_per_night INTEGER,
    stars INTEGER,
    distance_to_center REAL,
    amenities TEXT,
    rating REAL
)
""")

# Rooms table
cursor.execute("""
CREATE TABLE IF NOT EXISTS rooms (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    hotel_id INTEGER,
    room_type TEXT,
    price_multiplier REAL,
    available_rooms INTEGER
)
""")

# Reservations table
cursor.execute("""
CREATE TABLE IF NOT EXISTS reservations (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    hotel_id INTEGER,
    room_type TEXT,
    guest_name TEXT,
    checkin TEXT,
    checkout TEXT,
    total_price REAL,
    confirmation_code TEXT
)
""")


In [ ]:
hotels_data = [
    # PARIS
    ("Arc Luxury Paris", "paris", 220, 5, 0.8, "wifi,pool,gym,spa,breakfast", 4.7),
    ("Central Stay Paris", "paris", 150, 4, 0.5, "wifi,gym,breakfast", 4.3),
    ("Seine Boutique Hotel", "paris", 180, 4, 1.0, "wifi,spa,breakfast", 4.5),
    ("Paris Budget Inn", "paris", 95, 3, 1.5, "wifi", 4.0),
    ("Eiffel Grand Retreat", "paris", 310, 5, 0.6, "wifi,pool,gym,spa,breakfast,bar", 4.8),

    # NEW YORK
    ("NY Comfort Suites", "new york", 180, 4, 0.7, "wifi,gym,breakfast", 4.4),
    ("Manhattan Luxe Palace", "new york", 350, 5, 0.4, "wifi,pool,gym,spa,bar", 4.9),
    ("Times Square Stay", "new york", 200, 4, 0.3, "wifi,gym,bar", 4.5),
    ("Brooklyn Budget Lodge", "new york", 110, 3, 2.0, "wifi", 4.1),
    ("Central Park Executive", "new york", 275, 5, 0.6, "wifi,gym,spa,breakfast", 4.6),

    # DUBAI
    ("Dubai Grand Palace", "dubai", 300, 5, 1.2, "wifi,pool,gym,spa,bar", 4.8),
    ("Desert Pearl Hotel", "dubai", 210, 4, 1.5, "wifi,pool,gym", 4.4),
    ("Marina View Resort", "dubai", 260, 5, 0.9, "wifi,pool,gym,spa,breakfast", 4.7),
    ("Dubai Budget Stay", "dubai", 120, 3, 3.0, "wifi,breakfast", 4.0),
    ("Palm Royal Suites", "dubai", 400, 5, 0.5, "wifi,pool,gym,spa,private beach,bar", 4.9),
]

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    
    # Insert hotels
    cursor.executemany(
        "INSERT INTO hotels (name, city, price_per_night, stars, distance_to_center, amenities, rating) VALUES (?, ?, ?, ?, ?, ?, ?)",
        hotels_data
    )
    
    conn.commit() 

    # Verify insertion
    cursor.execute("SELECT id, name, city, price_per_night FROM hotels")
    hotels = cursor.fetchall()
    print("Hotels in DB:")
    for h in hotels:
        print(h)

In [ ]:
import sqlite3

DB = "arc_hotels.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    
    # Check hotels
    cursor.execute("SELECT id, name, city, price_per_night FROM hotels")
    hotels = cursor.fetchall()
    print("Hotels in DB:")
    for h in hotels:
        print(h)

    # Check rooms
    cursor.execute("SELECT * FROM rooms")
    rooms = cursor.fetchall()
    print("\nRooms in DB:")
    for r in rooms:
        print(r)

    # Check reservations
    cursor.execute("SELECT * FROM reservations")
    reservations = cursor.fetchall()
    print("\nReservations in DB:")
    for res in reservations:
        print(res)

In [41]:
room_types =[
    ("Standard", 1.0),
    ("Deluxe", 1.3),
    ("Suite", 1.8),
]

for hotel_id in range(1, 16):
    for room_type, multiplier in room_types:
        cursor.execute(
            """
              INSERT INTO rooms (hotel_id, room_type, price_multiplier, available_rooms)
        VALUES (?, ?, ?, ?)""", (hotel_id, room_type, multiplier, 10)
        )
conn.commit()

In [42]:
system_message = """
You are ArcHotels Booking Agent and Advisor.
You help customers find and book hotels that meets their budgets and constraints.

Important rules you must follow:

1. If the user has not yet provided check-in and check-out dates, ask for them.
2. Never attempt to search for hotels without BOTH check-in and check-out.
3. When calling the `search_hotels` tool, ALWAYS include:
   - city
   - max_price (total budget)
   - checkin
   - checkout
4. Treat max_price as the total budget for the whole stay, not per night.
5. Convert total budget into a per-night budget internally (max_price / nights).
6. Do not attempt to reason about hotels without calling the tool.
7. If data is missing (dates, budget), ask a clear question.
8. If booking is requested, call the `reserve_room` tool.
9. Do not invent hotel names, prices, or features.
Be concise and professional.
"""

In [43]:
def calculate_nights(checkin, checkout):
    checkin_date = datetime.strptime(checkin, "%Y-%m-%d")
    checkout_date = datetime.strptime(checkout, "%Y-%m-%d")
    return (checkout_date - checkin_date).days

In [44]:
def search_hotels(city, max_price=None, required_amenities=None,
                  min_stars=None, checkin=None, checkout=None):

    print(f"DATABASE TOOL CALLED: Searching hotels in {city}", flush=True)

    # Calculate nights
    nights = 1
    if checkin and checkout:
        try:
            nights = calculate_nights(checkin, checkout)
            if nights <= 0:
                nights = 1
        except:
            # fallback
            nights = 1

    # Compute price per night budget from total budget
    max_price_per_night = None
    if max_price is not None:
        max_price_per_night = max_price / nights

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        query = "SELECT * FROM hotels WHERE LOWER(city) = ?"
        params = [city.lower()]

        if min_stars:
            query += " AND stars >= ?"
            params.append(min_stars)

        cursor.execute(query, params)
        results = cursor.fetchall()

    filtered_results = []
    for hotel in results:
        price_per_night = hotel[3]

        # filter by per-night budget-i.e if max price exist and the hotel price per night greater than it, then skip this hotel
        if max_price_per_night and price_per_night > max_price_per_night:
            continue

        # filter amenities
        if required_amenities:
            hotel_amenities = hotel[6].lower().split(",")
            if not all(a.lower() in hotel_amenities for a in required_amenities):
                continue

        total_price = price_per_night * nights
        filtered_results.append((hotel, total_price))

    filtered_results.sort(key=lambda x: x[0][7], reverse=True)

    if not filtered_results:
        return f"No matching hotels found in {city} for {nights} night(s) within a total budget of ${max_price}."

    response = f"Top matching hotels in {city} for {nights} night(s):\n"
    for hotel, total_price in filtered_results[:3]:
        response += (
            f"{hotel[1]} | ${hotel[3]}/night | Total: ${total_price:.2f} | "
            f"{hotel[4]}⭐ | Rating {hotel[7]}\n"
        )

    return response

In [45]:
def get_hotel_details(hotel_name):
    print(f"DATABASE TOOL CALLED: Getting details for {hotel_name}", flush=True)
    
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        cursor.execute("SELECT * FROM hotels WHERE name = ?", (hotel_name,))
        hotel = cursor.fetchone()

        if not hotel:
            return "Hotel not found."

        cursor.execute(
            "SELECT room_type, price_multiplier, available_rooms FROM rooms WHERE hotel_id = ?",
            (hotel[0],)
        )
        rooms = cursor.fetchall()

    response = f"{hotel[1]} Details:\n"
    response += f"{hotel[4]}⭐ | Rating {hotel[7]} | ${hotel[3]}/night base price\n"
    response += f"Amenities: {hotel[6]}\nRooms:\n"

    for room in rooms:
        final_price = hotel[3] * room[1]
        response += f"{room[0]} - ${final_price:.2f} per night ({room[2]} available)\n"

    return response

In [46]:
def reserve_room(hotel_name, room_type, guest_name, checkin, checkout):
    print(f"DATABASE TOOL CALLED: Reserving room at {hotel_name}", flush=True)

    conn = sqlite3.connect(DB)
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM hotels WHERE name = ?", (hotel_name,))
    hotel = cursor.fetchone()

    if not hotel:
        return "Hotel not found."

    cursor.execute("""
        SELECT price_multiplier, available_rooms 
        FROM rooms 
        WHERE hotel_id = ? AND room_type = ?
    """, (hotel[0], room_type))

    room = cursor.fetchone()

    if not room or room[1] <= 0:
        return "Room type unavailable."

    nights = calculate_nights(checkin, checkout)
    total_price = hotel[3] * room[0] * nights

    confirmation_code = ''.join(random.choices(string.ascii_uppercase + string.digits, k=8))

    cursor.execute("""
        INSERT INTO reservations 
        (hotel_id, room_type, guest_name, checkin, checkout, total_price, confirmation_code)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (hotel[0], room_type, guest_name, checkin, checkout, total_price, confirmation_code))

    cursor.execute("""
        UPDATE rooms 
        SET available_rooms = available_rooms - 1
        WHERE hotel_id = ? AND room_type = ?
    """, (hotel[0], room_type))

    conn.commit()
    conn.close()

    return f"Reservation confirmed at {hotel_name}. Total: ${total_price:.2f}. Confirmation: {confirmation_code}"

In [ ]:
conn = sqlite3.connect(DB)
cursor = conn.cursor()

# Verify hotels
cursor.execute("SELECT hotel_id, room_type, guest_name, checkin, checkout, total_price, confirmation_code FROM reservations")
print(cursor.fetchall())


conn.close()

In [48]:
search_hotels_tool = {
    "name": "search_hotels",
    "description": "Search for hotels based on city, price, amenities, stars, and stay dates.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string"},
            "max_price": {"type": "number"},
            "required_amenities": {
                "type": "array",
                "items": {"type": "string"}
            },
            "min_stars": {"type": "number"},
            "checkin": {"type": "string"},
            "checkout": {"type": "string"}
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

get_details_tool = {
    "name": "get_hotel_details",
    "description": "Get detailed information about a specific hotel.",
    "parameters": {
        "type": "object",
        "properties": {
            "hotel_name": {"type": "string"}
        },
        "required": ["hotel_name"],
        "additionalProperties": False
    }
}

reserve_room_tool = {
    "name": "reserve_room",
    "description": "Reserve a room at a hotel.",
    "parameters": {
        "type": "object",
        "properties": {
            "hotel_name": {"type": "string"},
            "room_type": {"type": "string"},
            "guest_name": {"type": "string"},
            "checkin": {"type": "string"},
            "checkout": {"type": "string"}
        },
        "required": ["hotel_name", "room_type", "guest_name", "checkin", "checkout"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": search_hotels_tool},
    {"type": "function", "function": get_details_tool},
    {"type": "function", "function": reserve_room_tool},
]

In [49]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        if function_name == "search_hotels":
            result = search_hotels(**arguments)

        elif function_name == "get_hotel_details":
            result = get_hotel_details(**arguments)

        elif function_name == "reserve_room":
            result = reserve_room(**arguments)

        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return responses

In [50]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages =[{"role":"system", "content": system_message}] + history + [{"role":"user", "content": message}]

    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message_response = response.choices[0].message
        tool_responses = handle_tool_calls(message_response)

        messages.append(message_response)
        messages.extend(tool_responses)

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Hotelia (The ArcHotels Booking Agent and Advisor)"
).launch(
    inbrowser=True,
    auth=("admin", "arc2026")
)